# DLMDSPWP01: Written Assignment End-to-End Pipeline
IU International University of Applied Sciences

This notebook demonstrates the complete stepwise workflow for the DLMDSPWP01 assignment:
1. Environment and Package Initialization
2. Data Ingestion and Validation (Pandas)
3. Least-Squares Regression and Ideal Function Selection
4. Maximum Deviation and Threshold Derivation
5. Line-by-Line Test Data Classification (Generator Protocol)
6. Database Persistence via SQLAlchemy Core (SQLite Tables 1, 2, and 3)
7. Interactive Data Visualization (Bokeh Dashboard)
8. Automated Unit Testing (unittest Suite)
9. Version Control Workflow Documentation (Additional Task 1.3)

## Step 1: Environment and Package Initialization
Import required scientific libraries and reload the modular assignment package (Unit 1.5).

In [ ]:
import math
import os
import sys
import unittest
import pandas as pd
import sqlalchemy as db
from bokeh.plotting import figure, output_file, save
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, Band

# Ensure project root is available in module search path
if '..' not in sys.path:
    sys.path.insert(0, '..')
if '.' not in sys.path:
    sys.path.insert(0, '.')

import importlib
import assignment
import assignment.exceptions
import assignment.processors
import assignment.database
import assignment.visualizer

importlib.reload(assignment.exceptions)
importlib.reload(assignment.processors)
importlib.reload(assignment.database)
importlib.reload(assignment.visualizer)
importlib.reload(assignment)

from assignment import (
    AssignmentBaseException,
    DataLoadingError,
    DataValidationError,
    FunctionMatchingError,
    DatabaseOperationError,
    TrainingDatasetProcessor,
    IdealDatasetProcessor,
    TestDatasetProcessor,
    DatabaseManager,
    DataVisualizer,
)

print('Step 1 complete: Libraries imported and assignment package initialized.')

## Step 2: Exploratory Data Ingestion and Inspection
Load train.csv, ideal.csv, and test.csv to inspect dimensions, columns, and data types (Unit 3.2).

In [ ]:
train_path = '../data/train.csv' if os.path.exists('../data/train.csv') else 'data/train.csv'
ideal_path = '../data/ideal.csv' if os.path.exists('../data/ideal.csv') else 'data/ideal.csv'
test_path = '../data/test.csv' if os.path.exists('../data/test.csv') else 'data/test.csv'

train_df = pd.read_csv(train_path)
ideal_df = pd.read_csv(ideal_path)
test_df = pd.read_csv(test_path)

print(f'Training dataset: {train_df.shape[0]} rows, {train_df.shape[1]} columns')
print(f'Ideal functions dataset: {ideal_df.shape[0]} rows, {ideal_df.shape[1]} columns')
print(f'Test dataset: {test_df.shape[0]} rows, {test_df.shape[1]} columns')

print('\nTraining dataset preview (first 3 rows):')
print(train_df.head(3))

## Step 3: Mathematical Formulation and Ideal Function Selection
Select the four ideal functions that minimize the sum of squared deviations across all fifty candidate functions (Least-Squares criterion):

$$\text{SSE}_{k, j} = \sum_{i=1}^{N} (y_{\text{train}, k}^{(i)} - y_{\text{ideal}, j}^{(i)})^2$$

In [ ]:
train_processor = TrainingDatasetProcessor(train_path)
ideal_processor = IdealDatasetProcessor(ideal_path)

train_processor.load_data()
ideal_processor.load_data()

chosen_models = train_processor.select_best_ideal_functions(ideal_processor)

print(f"{'Training Func':<15}{'Chosen Ideal':<15}{'Sum Squared Dev':<20}{'Max Dev':<12}{'Threshold (sqrt(2))':<15}")
for t_col, info in chosen_models.items():
    print(f"{t_col:<15}{info['chosen_ideal_col']:<15}{info['sum_squared_deviation']:<20.4f}{info['max_deviation']:<12.4f}{info['threshold']:<15.4f}")

## Step 4: Line-by-Line Test Data Classification
Stream test points directly line-by-line from the CSV file using Python's generator protocol (Unit 2.3).
Map each point if its deviation from the chosen ideal function does not exceed $\text{max\_dev} \times \sqrt{2}$.

In [ ]:
test_processor = TestDatasetProcessor(test_path)

# Verify line-by-line streaming without bulk preloading
streamed_rows = list(test_processor.stream_test_data())
print(f'Total rows streamed line-by-line: {len(streamed_rows)}')

# Perform regression classification and mapping
mapped_records, all_summary = test_processor.map_test_data(ideal_processor, chosen_models)

print(f'Total test cases evaluated: {len(streamed_rows)}')
print(f'Test cases successfully assigned: {len(mapped_records)}')
print(f'Test cases unassigned: {len(streamed_rows) - len(mapped_records)}')

print('\nFirst 5 mapped records for Table 3:')
for record in mapped_records[:5]:
    print(record)

## Step 5: Database Persistence via SQLAlchemy Core
Compile SQLite database schema and populate Table 1 (training data), Table 2 (ideal functions), and Table 3 (mapped test data) per Section 1.2 specifications (Unit 3.5).

In [ ]:
db_output_path = '../outputs/assignment.db' if os.path.exists('../outputs') else 'outputs/assignment.db'
db_manager = DatabaseManager(db_output_path)

# 1. Compile schema
db_manager.compile_database_schema()

# 2. Persist datasets
db_manager.save_training_data(train_processor.get_dataframe())
db_manager.save_ideal_functions(ideal_processor.get_dataframe())
db_manager.save_test_data_mapping(mapped_records)

# 3. Verify record counts
count_t1 = db_manager.fetch_records_count('training_data')
count_t2 = db_manager.fetch_records_count('ideal_functions')
count_t3 = db_manager.fetch_records_count('test_data_mapping')

print(f'Table 1 (training_data) records: {count_t1}')
print(f'Table 2 (ideal_functions) records: {count_t2}')
print(f'Table 3 (test_data_mapping) records: {count_t3}')

## Step 6: Interactive Data Visualization with Bokeh
Generate an interactive HTML dashboard containing individual subplots for each chosen function pair with tolerance bands, training points, and assigned test cases with hover tooltips (Unit 3.4.3).

In [ ]:
html_output_path = '../outputs/visualization.html' if os.path.exists('../outputs') else 'outputs/visualization.html'
visualizer = DataVisualizer(html_output_path)

saved_html_path = visualizer.generate_visualization(
    train_processor.get_dataframe(),
    ideal_processor.get_dataframe(),
    chosen_models,
    all_summary
)

print(f'Interactive visualization saved: {saved_html_path}')
print(f'Dashboard size: {os.path.getsize(saved_html_path)} bytes')

## Step 7: Automated Unit Testing Verification
Execute automated test cases using Python's standard unittest framework covering custom exceptions, loaders, math, generator streaming, and SQLite persistence (Unit 5.3).

In [ ]:
from tests.test_assignment import (
    TestCustomExceptions,
    TestDatasetLoaders,
    TestFunctionMatchingLogic,
    TestTestDataMapping,
    TestDatabaseManager,
    TestVisualizer,
    TestValidationEdgeCases,
)

suite = unittest.TestSuite()
test_classes = [
    TestCustomExceptions,
    TestDatasetLoaders,
    TestFunctionMatchingLogic,
    TestTestDataMapping,
    TestDatabaseManager,
    TestVisualizer,
    TestValidationEdgeCases,
]
for cls in test_classes:
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
test_results = runner.run(suite)

print(f"\nUnit test execution complete: {test_results.testsRun} tests run, "
      f"{len(test_results.failures)} failures, {len(test_results.errors)} errors.")


## Step 8: Version Control Workflow Documentation (Additional Task 1.3)
Git commands to clone the develop branch, implement changes on a feature branch, and open a pull request (Unit 6):

```bash
# 1. Clone develop branch locally
git clone -b develop https://github.com/ainaomotayo/DLMDSPWP01.git
cd DLMDSPWP01

# 2. Create feature branch
git checkout -b feature/model-matching

# 3. Stage changes
git add assignment/ main.py tests/ requirements.txt notebooks/

# 4. Commit changes
git commit -m "Implement ideal function matching pipeline and test suite"

# 5. Push feature branch to remote
git push origin feature/model-matching

# 6. Open Pull Request on GitHub targeting develop branch for peer review
```